# Resampling-Aware Inflation Decomposition (RAID): A Leakage-Controlled Benchmark for Online Purchase-Intention Prediction

**Wagino, Muhammad Subhan, Silvia Ratna, M. Muflih** — Universitas Islam Kalimantan Muhammad Arsyad Al Banjari, Banjarmasin, Indonesia
*DECODE: Jurnal Pendidikan Teknologi Informasi* (revised manuscript no. 1970)

This single notebook reproduces every table and figure of the revised manuscript and its supplementary material.
Output files are named after the table or figure they feed.

| Section | Produces | Approx. time (Colab CPU) |
|---|---|---|
| 1 | Table 6 — model comparison on the test set and by repeated 5×5 CV | 15–25 min |
| 2 | Table 7 — one-factor ablation and paired tests; Supplementary Tables S4, S5 | 10–15 min |
| 3 | Table 8, Figures 2–3 — SHAP (grouped by original variable, beeswarm, waterfall); Table S7 | 3 min |
| 4 | Tables 9–10 — nested 5×5 CV and bootstrap 95% CI; Table S8 (threshold, F1 variants) | 20–40 min |
| 5 | Figure 4, Table S9 — recalibration | 2 min |
| 6 | Tables 11–12, Figure 5 — RAID, 30 seeds × 4 classifiers (**resumable**); Tables S1–S3 | 30–60 min |
| 7 | Table S6 — deduplicated data | 2 min |

**Run locally:** put the UCI CSV in `Dataset/` (any file name), then *Run All*. Results go to `Results/`.
**Run in Google Colab:** *Run all*; cell 0.1 installs the pinned versions and restarts the runtime once; then *Run all* again.
Edit `DATA_PATH` in cell 0.2 if your CSV is elsewhere on Google Drive.
Set entries of `RUN` in cell 0.2 to `False` to skip sections. A single seed (42) controls the split, tuning,
feature selection, SHAP sampling and bootstrap; RAID uses seeds 0–29.

## 0. Setup

In [ ]:
# 0.1 Pinned environment (same versions as the submitted manuscript).
# First "Run all": installs missing versions and restarts the Colab runtime automatically (the run stops here).
# Second "Run all": versions are already correct, so everything runs to the end.
import importlib.metadata as _md, subprocess, sys, os
PINS = {"numpy": "2.1.3", "pandas": "2.2.3", "scikit-learn": "1.6.1", "xgboost": "3.4.1",
        "catboost": "1.2.10", "imbalanced-learn": "0.14.2", "shap": "0.52.0"}
def _ver(p):
    try: return _md.version(p)
    except _md.PackageNotFoundError: return None
need = [f"{k}=={v}" for k, v in PINS.items() if _ver(k) != v]
if not need:
    print("Pinned versions OK - continue.")
elif "google.colab" in sys.modules or os.path.exists("/content"):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *need], check=True)
    print("Installed:", need, "\n>>> The runtime will now restart. When it reconnects, choose Runtime -> Run all again. <<<")
    os.kill(os.getpid(), 9)
else:
    print("WARNING: versions differ from the manuscript:", need)

In [ ]:
# 0.2 Configuration — edit only this cell
import os, sys
from pathlib import Path
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DATA_PATH = Path(os.environ.get("RAID_DATA", "/content/drive/MyDrive/datasetonlines/DataSet_online_shoppers_intention.csv"))
    OUT_DIR   = Path(os.environ.get("RAID_OUT",  "/content/drive/MyDrive/datasetonlines/Results"))
else:
    DATA_PATH = Path(os.environ.get("RAID_DATA", "Dataset/online_shoppers_intention.csv"))
    OUT_DIR   = Path(os.environ.get("RAID_OUT",  "Results"))
    if not DATA_PATH.exists():                      # accept any CSV placed in Dataset/
        cands = sorted(Path("Dataset").glob("*.csv"))
        assert cands, "Put the UCI Online Shoppers CSV in Dataset/ (see Dataset/README.md)."
        DATA_PATH = cands[0]

SEED = 42
TEST_SIZE = 0.20
N_BOOT = 2000                      # bootstrap resamples (Table 7 paired tests, Table 9)
CV_SPLITS, CV_REPEATS = 5, 5       # Table 6 repeated stratified CV on the training set
ABL_CV_REPEATS = 3                 # Table 7 repeated CV for the ablation
NESTED_ITER = 30                   # Table 9 nested CV: RandomizedSearch candidates per outer fold
RAID_SEEDS = [int(s) for s in os.environ.get("RAID_SEEDS", ",".join(map(str, range(30)))).split(",")]
RAID_MODELS = ["XGBoost", "RandomForest", "CatBoost", "LogisticRegression"]
RUN = dict(compare=True, ablation=True, shap=True, final_eval=True, nested_cv=True,
           calibration=True, raid=True, dedup=True)

FIG = OUT_DIR / "figures"
OUT_DIR.mkdir(parents=True, exist_ok=True); FIG.mkdir(exist_ok=True)
print("IN_COLAB:", IN_COLAB, "| data:", DATA_PATH, "| out:", OUT_DIR)

In [ ]:
# 0.3 Imports, constants and helpers
import warnings, json, time, platform
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib
import matplotlib.pyplot as plt
from scipy import sparse, stats
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.feature_selection import SelectKBest, mutual_info_classif
from sklearn.model_selection import train_test_split, RepeatedStratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
                             average_precision_score, brier_score_loss, log_loss)
from sklearn.calibration import calibration_curve
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
import sklearn, xgboost, catboost, imblearn, shap

TARGET = "Revenue"
CAT = ["Month", "OperatingSystems", "Browser", "Region", "TrafficType", "VisitorType", "Weekend"]
NUM = ["Administrative", "Administrative_Duration", "Informational", "Informational_Duration",
       "ProductRelated", "ProductRelated_Duration", "BounceRates", "ExitRates", "PageValues", "SpecialDay"]
TEMPORAL = ["Month", "Weekend", "SpecialDay"]
TUNED = dict(n_estimators=617, max_depth=6, learning_rate=0.02317246825143534, subsample=0.7359596102001048,
             colsample_bytree=0.7410275425336676, gamma=0.1828687548948174, min_child_weight=1,
             reg_alpha=0, reg_lambda=1)
BASE = dict(n_estimators=300, max_depth=6, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8)
METRICS = ["Accuracy", "Precision", "Recall", "F1", "ROC_AUC", "PR_AUC"]

def prep(num=NUM, cat=CAT, scale=False):
    ns = [("imputer", SimpleImputer(strategy="median"))] + ([("scaler", StandardScaler())] if scale else [])
    return ColumnTransformer([("num", Pipeline(ns), num),
                              ("cat", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")),
                                                ("onehot", OneHotEncoder(handle_unknown="ignore"))]), cat)])

def xgb(params, spw=None, seed=SEED):
    return XGBClassifier(objective="binary:logistic", eval_metric="logloss", random_state=seed,
                         scale_pos_weight=spw, **params)

def spw_of(y_): y_ = np.asarray(y_); return (y_ == 0).sum() / (y_ == 1).sum()

def metrics(y_true, prob, thr=0.5, w=None):
    pred = (np.asarray(prob) >= thr).astype(int)
    return dict(Accuracy=accuracy_score(y_true, pred, sample_weight=w),
                Precision=precision_score(y_true, pred, zero_division=0, sample_weight=w),
                Recall=recall_score(y_true, pred, zero_division=0, sample_weight=w),
                F1=f1_score(y_true, pred, zero_division=0, sample_weight=w),
                ROC_AUC=roc_auc_score(y_true, prob, sample_weight=w),
                PR_AUC=average_precision_score(y_true, prob, sample_weight=w))

def save(df, name):
    p = OUT_DIR / name; df.to_csv(p, index=False); print("saved ->", p); return df

env = pd.DataFrame([("Python", platform.python_version()), ("numpy", np.__version__), ("pandas", pd.__version__),
                    ("scikit-learn", sklearn.__version__), ("xgboost", xgboost.__version__),
                    ("catboost", catboost.__version__), ("imbalanced-learn", imblearn.__version__),
                    ("shap", shap.__version__), ("matplotlib", matplotlib.__version__)], columns=["Component", "Version"])
save(env, "environment.csv"); print(env.to_string(index=False))
expected = {"numpy": "2.1.3", "scikit-learn": "1.6.1", "xgboost": "3.4.1", "imbalanced-learn": "0.14.2"}
for k, v in expected.items():
    got = env.set_index("Component").loc[k, "Version"]
    if got != v: print(f"WARNING: {k} {got} != pinned {v}; small numeric differences are possible.")

In [ ]:
# 0.4 Load data and create the stratified 80/20 split (seed 42) used throughout
df = pd.read_csv(DATA_PATH)
df[TARGET] = df[TARGET].astype(str).str.upper().map({"TRUE": 1, "FALSE": 0, "1": 1, "0": 0}).astype(int)
for c in CAT: df[c] = df[c].astype(str)        # coded identifiers are categorical, not ordinal
X, y = df.drop(columns=[TARGET]), df[TARGET]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=TEST_SIZE, stratify=y, random_state=SEED)
SPW = spw_of(y_train)
print(df.shape, "| train", X_train.shape, "| test", X_test.shape, "| test positives", int(y_test.sum()), "| spw", round(SPW, 4))
assert df.shape == (12330, 18) and int(y_test.sum()) == 382, "Dataset or split differs from the manuscript."

def final_pipeline(spw, k=20):
    # Final model: tuned XGBoost + mutual-information feature selection (k = 20), scale_pos_weight from the training data
    return Pipeline([("p", prep()), ("fs", SelectKBest(mutual_info_classif, k=k)), ("m", xgb(TUNED, spw))])

def catboost_model(seed=SEED):
    return CatBoostClassifier(iterations=300, depth=6, learning_rate=0.05, loss_function="Logloss", eval_metric="AUC",
                              auto_class_weights="Balanced", verbose=0, random_seed=seed)

final = final_pipeline(SPW).fit(X_train, y_train)          # reused in Sections 3 and 4
y_prob_final = final.predict_proba(X_test)[:, 1]
print("Final model test F1 (manuscript 0.6576):", round(metrics(y_test, y_prob_final)["F1"], 4))

## 1. Model comparison (Table 6)
**1a. Test set.** Four fixed, untuned classifiers and the final model, all fitted on the training set only.
**1b. Repeated stratified 5×5 CV on the training set**, with the corrected resampled t-test (Nadeau & Bengio, 2003)
against the final model, so that model choice does not rely on the test set.

In [ ]:
if RUN["compare"]:
    models = {"Logistic Regression": Pipeline([("p", prep(scale=True)), ("m", LogisticRegression(max_iter=2000, class_weight="balanced", random_state=SEED))]),
              "Random Forest": Pipeline([("p", prep()), ("m", RandomForestClassifier(n_estimators=300, class_weight="balanced", random_state=SEED, n_jobs=-1))]),
              "XGBoost (untuned)": Pipeline([("p", prep()), ("m", xgb(BASE, SPW))])}
    def hard(name, mdl, Xt):
        # class labels from .predict() (as in the submitted results); ranking metrics from probabilities
        pred, prob = np.asarray(mdl.predict(Xt)).astype(int).ravel(), mdl.predict_proba(Xt)[:, 1]
        return {"Model": name, **metrics(y_test, prob), "Accuracy": accuracy_score(y_test, pred),
                "Precision": precision_score(y_test, pred, zero_division=0), "Recall": recall_score(y_test, pred, zero_division=0),
                "F1": f1_score(y_test, pred, zero_division=0)}
    rows = [hard(name, pipe.fit(X_train, y_train), X_test) for name, pipe in models.items()]
    rows.append(hard("CatBoost", catboost_model().fit(X_train, y_train, cat_features=CAT), X_test))
    rows.append(hard("XGBoost (tuned + FS, final)", final, X_test))
    t6 = pd.DataFrame(rows); t6[METRICS] = t6[METRICS].round(4); save(t6, "table06_test_set.csv"); display(t6)

In [ ]:
def cv_models():
    return {"Logistic Regression": lambda ya: Pipeline([("p", prep(scale=True)), ("m", LogisticRegression(max_iter=2000, class_weight="balanced", random_state=SEED))]),
            "Random Forest": lambda ya: Pipeline([("p", prep()), ("m", RandomForestClassifier(n_estimators=300, class_weight="balanced", random_state=SEED, n_jobs=-1))]),
            "XGBoost (untuned)": lambda ya: Pipeline([("p", prep()), ("m", xgb(BASE, spw_of(ya)))]),
            "XGBoost (tuned + FS, final)": lambda ya: Pipeline([("p", prep()), ("fs", SelectKBest(mutual_info_classif, k=20)), ("m", xgb(TUNED, spw_of(ya)))]),
            "CatBoost": None}

def corrected_ttest(d, n_train, n_test, k):
    d = np.asarray(d); v = d.var(ddof=1)
    t = d.mean() / np.sqrt((1 / k + n_test / n_train) * v) if v > 0 else np.inf
    return t, 2 * stats.t.sf(abs(t), df=k - 1)

if RUN["compare"]:
    rkf = RepeatedStratifiedKFold(n_splits=CV_SPLITS, n_repeats=CV_REPEATS, random_state=SEED); rows = []
    for f, (a, b) in enumerate(rkf.split(X_train, y_train)):
        Xa, Xb, ya, yb = X_train.iloc[a], X_train.iloc[b], y_train.iloc[a], y_train.iloc[b]
        for name, make in cv_models().items():
            if name == "CatBoost":
                mdl = catboost_model().fit(Xa, ya, cat_features=CAT)
            else:
                mdl = make(ya).fit(Xa, ya)
            rows.append({"Model": name, "fold": f, **metrics(yb, mdl.predict_proba(Xb)[:, 1])})
        print(f"fold {f + 1}/{CV_SPLITS * CV_REPEATS}")
    cvm = pd.DataFrame(rows); save(cvm, "table06_cv_folds.csv")
    summ = cvm.groupby("Model")[METRICS].agg(["mean", "std"]).round(4); summ.columns = [f"{a}_{b}" for a, b in summ.columns]
    ref = "XGBoost (tuned + FS, final)"; k = CV_SPLITS * CV_REPEATS; n_te = len(X_train) // CV_SPLITS; n_tr = len(X_train) - n_te
    tests = []
    for name in cvm.Model.unique():
        if name == ref: continue
        for m in ["F1", "PR_AUC", "ROC_AUC"]:
            d = cvm[cvm.Model == ref].sort_values("fold")[m].values - cvm[cvm.Model == name].sort_values("fold")[m].values
            t, p = corrected_ttest(d, n_tr, n_te, k)
            tests.append({"Reference": ref, "Compared": name, "Metric": m, "MeanDiff": round(d.mean(), 4), "t": round(t, 3), "p": round(p, 4)})
    save(summ.reset_index(), "table06_cv_summary.csv"); save(pd.DataFrame(tests), "table06_cv_tests.csv")
    display(summ); display(pd.DataFrame(tests))

## 2. One-factor ablation (Table 7) and paired tests
Single reference = **tuned XGBoost, all features, scale_pos_weight**; each variant changes exactly one factor.
The untuned configuration is reported separately. Test set plus repeated stratified 5×3 CV on the training set.
Differences on the test set are tested with a paired bootstrap (2,000 resamples) and an exact McNemar test.
The same block gives the feature-selection k-sensitivity (Supplementary Table S5) and the CatBoost ablation (Table S4).

In [ ]:
def ablation_configs():
    nt = [c for c in NUM if c not in TEMPORAL]; ct = [c for c in CAT if c not in TEMPORAL]
    npv = [c for c in NUM if c != "PageValues"]
    return {
      "Reference: tuned XGBoost, all features": (Pipeline([("p", prep()), ("m", xgb(TUNED, "spw"))]), None),
      "+ Feature selection (MI, k=20)":         (Pipeline([("p", prep()), ("fs", SelectKBest(mutual_info_classif, k=20)), ("m", xgb(TUNED, "spw"))]), None),
      "+ SMOTE instead of scale_pos_weight":    (ImbPipeline([("p", prep()), ("s", SMOTE(random_state=SEED)), ("m", xgb(TUNED, None))]), None),
      "- Temporal features (Month, Weekend, SpecialDay)": (Pipeline([("p", prep(nt, ct)), ("m", xgb(TUNED, "spw"))]), TEMPORAL),
      "- PageValues":                           (Pipeline([("p", prep(npv, CAT)), ("m", xgb(TUNED, "spw"))]), ["PageValues"]),
      "Untuned XGBoost (manuscript 'Baseline')": (Pipeline([("p", prep()), ("m", xgb(BASE, "spw"))]), None),
    }

def fit_cfg(pipe, drop, Xa, ya):
    p = clone(pipe)
    if p.named_steps["m"].get_params()["scale_pos_weight"] == "spw":
        p.named_steps["m"].set_params(scale_pos_weight=spw_of(ya))
    Xa = Xa.drop(columns=drop) if drop else Xa
    return p.fit(Xa, ya)

def predict_cfg(p, drop, Xb):
    return p.predict_proba(Xb.drop(columns=drop) if drop else Xb)[:, 1]

if RUN["ablation"]:
    holdout_prob, rows, cvrows = {}, [], []
    for name, (pipe, drop) in ablation_configs().items():
        p = fit_cfg(pipe, drop, X_train, y_train); pr = predict_cfg(p, drop, X_test)
        holdout_prob[name] = pr; rows.append({"Setting": name, **metrics(y_test, pr)})
        rkf = RepeatedStratifiedKFold(n_splits=CV_SPLITS, n_repeats=ABL_CV_REPEATS, random_state=SEED)
        for f, (a, b) in enumerate(rkf.split(X_train, y_train)):
            q = fit_cfg(pipe, drop, X_train.iloc[a], y_train.iloc[a])
            cvrows.append({"Setting": name, "fold": f, **metrics(y_train.iloc[b], predict_cfg(q, drop, X_train.iloc[b]))})
        print("done:", name)
    abl = pd.DataFrame(rows); abl[METRICS] = abl[METRICS].round(4)
    save(abl, "table07_ablation_test.csv")
    cv = pd.DataFrame(cvrows); save(cv, "table07_ablation_cv_folds.csv")
    summ = cv.groupby("Setting", sort=False)[["F1", "ROC_AUC", "PR_AUC"]].agg(["mean", "std"]).round(4)
    summ.columns = [f"{a}_{b}" for a, b in summ.columns]; save(summ.reset_index(), "table07_ablation_cv_summary.csv")
    display(abl); display(summ)

In [ ]:
def paired_bootstrap(y_true, pa, pb, n=N_BOOT, seed=SEED):
    rng = np.random.default_rng(seed); yt = np.asarray(y_true); N = len(yt); d = {m: [] for m in ["F1", "PR_AUC", "ROC_AUC"]}
    for _ in range(n):
        i = rng.integers(0, N, N)
        if yt[i].min() == yt[i].max(): continue
        ma, mb = metrics(yt[i], pa[i]), metrics(yt[i], pb[i])
        for m in d: d[m].append(ma[m] - mb[m])
    out = {}
    for m, v in d.items():
        v = np.array(v); full = metrics(yt, pa)[m] - metrics(yt, pb)[m]
        out[f"d{m}"] = round(full, 4); out[f"d{m}_CI"] = f"[{np.percentile(v, 2.5):.4f}; {np.percentile(v, 97.5):.4f}]"
    return out

def mcnemar_exact(y_true, pa, pb):
    yt = np.asarray(y_true); ca = (pa >= .5).astype(int) == yt; cb = (pb >= .5).astype(int) == yt
    b, c = int((ca & ~cb).sum()), int((~ca & cb).sum())
    p = stats.binomtest(min(b, c), b + c, 0.5).pvalue if b + c else 1.0
    return {"A_right_B_wrong": b, "A_wrong_B_right": c, "McNemar_p": round(p, 4)}

if RUN["ablation"]:
    P = holdout_prob; ref = "Reference: tuned XGBoost, all features"
    # k-sensitivity predictions (same pipeline as the manuscript's final model, varying k)
    for k in [10, 15, 25, 30]:
        p = Pipeline([("p", prep()), ("fs", SelectKBest(mutual_info_classif, k=k)), ("m", xgb(TUNED, SPW))]).fit(X_train, y_train)
        P[f"FS k={k}"] = p.predict_proba(X_test)[:, 1]
    cbm = catboost_model().fit(X_train, y_train, cat_features=CAT)
    P["CatBoost (Table 6)"] = cbm.predict_proba(X_test)[:, 1]
    fs20 = "+ Feature selection (MI, k=20)"; untuned = "Untuned XGBoost (manuscript 'Baseline')"
    pairs = [(fs20, ref), (fs20, untuned), (ref, untuned), (untuned, "CatBoost (Table 6)")] + \
            [(f"FS k={k}", fs20) for k in [10, 15, 25, 30]] + \
            [(n, ref) for n in ["+ SMOTE instead of scale_pos_weight", "- Temporal features (Month, Weekend, SpecialDay)", "- PageValues"]]
    rows = [{"A": a, "B": b, **paired_bootstrap(y_test, P[a], P[b]), **mcnemar_exact(y_test, P[a], P[b])} for a, b in pairs]
    paired = save(pd.DataFrame(rows), "table07_paired_tests.csv"); display(paired)
    ksens = pd.DataFrame([{"k": k, **metrics(y_test, P[f"FS k={k}"] if k != 20 else P[fs20])} for k in [10, 15, 20, 25, 30]])
    ksens[METRICS] = ksens[METRICS].round(4); save(ksens, "S5_k_sensitivity_test.csv"); display(ksens)

In [ ]:
# Supplementary Table S4 - CatBoost confirmation ablation (native categoricals, same split)
if RUN["ablation"]:
    rows = []
    for name, drop in [("CatBoost, all features", []), ("CatBoost - temporal features", TEMPORAL), ("CatBoost - PageValues", ["PageValues"])]:
        cats = [c for c in CAT if c not in drop]
        m = catboost_model().fit(X_train.drop(columns=drop), y_train, cat_features=cats)
        rows.append({"Setting": name, **metrics(y_test, m.predict_proba(X_test.drop(columns=drop))[:, 1])})
    s4 = pd.DataFrame(rows); s4[METRICS] = s4[METRICS].round(4); save(s4, "S4_catboost_ablation.csv"); display(s4)

## 3. SHAP explanation of the final model (Table 8, Figures 2–3, Table S7)
TreeExplainer on 500 test sessions (seed 42); values in log-odds. Grouped importance sums the SHAP values of all
one-hot columns of the same original variable per session and then takes the mean absolute value.
Figure 3 explains the first correctly predicted purchasing session of the test set.

In [ ]:
if RUN["shap"]:
    names = final.named_steps["p"].get_feature_names_out()[final.named_steps["fs"].get_support()]
    Xs = final.named_steps["fs"].transform(final.named_steps["p"].transform(X_test))
    Xs = pd.DataFrame(Xs.toarray() if sparse.issparse(Xs) else Xs, columns=names)
    sample = Xs.sample(n=min(500, len(Xs)), random_state=SEED)
    sv = shap.TreeExplainer(final.named_steps["m"]).shap_values(sample)
    enc = pd.DataFrame({"Feature": names, "MeanAbsSHAP": np.abs(sv).mean(0)}).sort_values("MeanAbsSHAP", ascending=False)
    def original(n):
        n = n.split("__", 1)[1]
        return next((c for c in CAT if n.startswith(c + "_")), n)
    groups = pd.Series([original(n) for n in names])
    grouped = pd.DataFrame(sv, columns=names).T.groupby(groups.values).sum().T
    grp = pd.DataFrame({"Variable": grouped.columns, "MeanAbsSHAP_grouped": np.abs(grouped.values).mean(0),
                        "n_encoded_columns": [int((groups == g).sum()) for g in grouped.columns]}).sort_values("MeanAbsSHAP_grouped", ascending=False)
    save(enc.round(6), "S7_shap_encoded.csv"); save(grp.round(6), "table08_shap_grouped.csv")
    plt.figure(); shap.summary_plot(sv, sample, show=False); plt.tight_layout(); plt.savefig(FIG / "figure2_shap_beeswarm.png", dpi=300, bbox_inches="tight"); plt.show()
    plt.figure(figsize=(6, 4)); g = grp.iloc[::-1]; plt.barh(g["Variable"], g["MeanAbsSHAP_grouped"])
    plt.xlabel("Mean |SHAP| (log-odds), grouped by original variable"); plt.tight_layout()
    plt.savefig(FIG / "table08_shap_grouped.png", dpi=300); plt.show(); display(enc.head(10)); display(grp)

In [ ]:
# Figure 3 - local waterfall for the first true positive of the test set
if RUN["shap"]:
    Xs_all = final.named_steps["fs"].transform(final.named_steps["p"].transform(X_test))
    Xs_all = pd.DataFrame(Xs_all.toarray() if sparse.issparse(Xs_all) else Xs_all, columns=names)
    tp = np.where((y_test.to_numpy() == 1) & (y_prob_final >= 0.5))[0]
    pos = int(tp[0]) if len(tp) else int(np.argmax(y_prob_final))
    local = shap.TreeExplainer(final.named_steps["m"])(Xs_all.iloc[[pos]])
    print("test position", pos, "| predicted probability", round(float(y_prob_final[pos]), 4))
    loc = pd.DataFrame({"Feature": names, "FeatureValue": Xs_all.iloc[pos].values, "SHAPValue": local.values[0]})
    loc = loc.reindex(loc.SHAPValue.abs().sort_values(ascending=False).index); save(loc.round(6), "figure3_shap_local.csv")
    plt.figure(); shap.plots.waterfall(local[0], max_display=15, show=False); plt.tight_layout()
    plt.savefig(FIG / "figure3_shap_waterfall.png", dpi=300, bbox_inches="tight"); plt.show()

## 4. Final model: threshold, nested CV and bootstrap 95% CI (Tables 9–10, Table S8)
**Threshold (Table S8).** A validation partition (20% of the training set, stratified, seed 42) selects the F1-optimal
threshold on the grid 0.10–0.90; it is then applied once to the test set.
**Bootstrap (Table 10).** 2,000 percentile-bootstrap resamples of the test set (seed 42; resamples with one class only
are redrawn); the model is not refitted, so the interval reflects test-set sampling only.
**Nested CV (Table 9).** Outer stratified 5-fold CV on all data; inside each outer training fold, preprocessing,
feature selection and a 30-candidate RandomizedSearchCV (inner 5-fold, F1) are refitted.

In [ ]:
def bootstrap_ci(y_true, prob, thr=0.5, n=N_BOOT, seed=SEED):
    rng = np.random.default_rng(seed); yt, pr = np.asarray(y_true), np.asarray(prob); N = len(yt); recs = []
    while len(recs) < n:
        i = rng.integers(0, N, N)
        if len(np.unique(yt[i])) < 2: continue
        recs.append({**metrics(yt[i], pr[i], thr), "Brier": brier_score_loss(yt[i], pr[i])})
    bs = pd.DataFrame(recs); point = {**metrics(yt, pr, thr), "Brier": brier_score_loss(yt, pr)}
    return pd.DataFrame([{"Metric": m, "Point": point[m], "CI_lower": bs[m].quantile(.025), "CI_upper": bs[m].quantile(.975)}
                         for m in bs.columns]).round(4)

if RUN["final_eval"]:
    # Threshold selection on a validation partition of the training set
    X_dev, X_val, y_dev, y_val = train_test_split(X_train, y_train, test_size=TEST_SIZE, stratify=y_train, random_state=SEED)
    pv = final_pipeline(spw_of(y_dev)).fit(X_dev, y_dev).predict_proba(X_val)[:, 1]
    thr = pd.DataFrame([{"Threshold": round(float(t), 2), **metrics(y_val, pv, t)} for t in np.arange(0.10, 0.91, 0.05)])
    thr = thr.sort_values("F1", ascending=False).reset_index(drop=True); best_t = float(thr.loc[0, "Threshold"])
    save(thr[["Threshold", "Accuracy", "Precision", "Recall", "F1"]].round(4), "S8_threshold_validation.csv")
    print("validation-selected threshold:", best_t)
    # Test-set estimates and bootstrap CI at 0.50 and at the selected threshold
    ci = pd.concat([bootstrap_ci(y_test, y_prob_final, 0.5).assign(Setting="Default threshold (0.50)"),
                    bootstrap_ci(y_test, y_prob_final, best_t).assign(Setting=f"Validation-selected threshold ({best_t:.2f})")])
    save(ci[["Setting", "Metric", "Point", "CI_lower", "CI_upper"]], "table10_bootstrap_ci.csv"); display(ci)
    pred = (y_prob_final >= 0.5).astype(int)
    f1v = pd.DataFrame([("Accuracy", accuracy_score(y_test, pred)), ("F1 (minority / positive class)", f1_score(y_test, pred)),
                        ("F1 (macro average)", f1_score(y_test, pred, average="macro")),
                        ("F1 (weighted average)", f1_score(y_test, pred, average="weighted"))], columns=["Metric", "Value"]).round(4)
    save(f1v, "S8_f1_variants.csv"); display(f1v)

In [ ]:
from sklearn.model_selection import StratifiedKFold, RandomizedSearchCV
from scipy.stats import randint, uniform
if RUN["nested_cv"]:
    outer = StratifiedKFold(5, shuffle=True, random_state=SEED); inner = StratifiedKFold(5, shuffle=True, random_state=SEED)
    dist = {"m__n_estimators": randint(300, 901), "m__max_depth": randint(4, 9), "m__learning_rate": uniform(0.01, 0.08),
            "m__subsample": uniform(0.7, 0.3), "m__colsample_bytree": uniform(0.6, 0.4), "m__min_child_weight": randint(1, 6),
            "m__gamma": uniform(0.0, 0.6), "m__reg_lambda": [1, 2, 5, 10], "m__reg_alpha": [0, 0.1, 0.5, 1.0]}
    rows, best = [], []
    for f, (a, b) in enumerate(outer.split(X, y), start=1):
        pipe = Pipeline([("p", prep()), ("fs", SelectKBest(mutual_info_classif, k=20)), ("m", xgb({}, spw_of(y.iloc[a])))])
        s = RandomizedSearchCV(pipe, dist, n_iter=NESTED_ITER, scoring="f1", cv=inner, n_jobs=-1, random_state=SEED, refit=True)
        s.fit(X.iloc[a], y.iloc[a])
        rows.append({"Fold": f, "Inner_best_CV_F1": s.best_score_, **metrics(y.iloc[b], s.predict_proba(X.iloc[b])[:, 1])})
        best.append({"Fold": f, **{k.replace("m__", ""): v for k, v in s.best_params_.items()}}); print(f"outer fold {f}/5 done")
    nf = pd.DataFrame(rows).round(4); save(nf, "table09_nested_cv_folds.csv"); save(pd.DataFrame(best), "table09_nested_cv_best_params.csv")
    ns = pd.DataFrame({"Metric": METRICS, "Mean": [nf[m].mean() for m in METRICS], "SD": [nf[m].std() for m in METRICS]}).round(4)
    save(ns, "table09_nested_cv_summary.csv"); display(ns)

## 5. Recalibration (Figure 4, Supplementary Table S9)
The model is trained on the development partition (the same split used for threshold selection); Platt scaling and
isotonic regression are fitted on the validation partition and evaluated on the untouched test set.

In [ ]:
def ece(y_true, prob, n_bins=10):
    y_true, prob = np.asarray(y_true), np.asarray(prob)
    edges = np.unique(np.quantile(prob, np.linspace(0, 1, n_bins + 1)))
    idx = np.clip(np.searchsorted(edges, prob, side="right") - 1, 0, len(edges) - 2)
    return float(sum(abs(prob[idx == b].mean() - y_true[idx == b].mean()) * (idx == b).mean() for b in range(len(edges) - 1) if (idx == b).any()))

if RUN["calibration"]:
    X_dev, X_val, y_dev, y_val = train_test_split(X_train, y_train, test_size=TEST_SIZE, stratify=y_train, random_state=SEED)
    print("development", X_dev.shape, "| validation", X_val.shape)
    m = Pipeline([("p", prep()), ("fs", SelectKBest(mutual_info_classif, k=20)), ("m", xgb(TUNED, spw_of(y_dev)))]).fit(X_dev, y_dev)
    pv, pt = m.predict_proba(X_val)[:, 1], m.predict_proba(X_test)[:, 1]
    logit = lambda p: np.log(np.clip(p, 1e-6, 1 - 1e-6) / (1 - np.clip(p, 1e-6, 1 - 1e-6)))
    platt = LogisticRegression(C=1e6).fit(logit(pv).reshape(-1, 1), y_val)
    iso = IsotonicRegression(out_of_bounds="clip").fit(pv, y_val)
    variants = {"Uncalibrated": pt, "Platt": platt.predict_proba(logit(pt).reshape(-1, 1))[:, 1], "Isotonic": iso.predict(pt)}
    rows, curves = [], []
    for k, p in variants.items():
        rows.append({"Variant": k, "Brier": brier_score_loss(y_test, p), "LogLoss": log_loss(y_test, np.clip(p, 1e-6, 1 - 1e-6)),
                     "ECE_quantile10": ece(y_test, p), **metrics(y_test, p)})
        for s in ["quantile", "uniform"]:
            ft, mp = calibration_curve(y_test, p, n_bins=10, strategy=s)
            curves += [{"Variant": k, "binning": s, "MeanPredicted": a, "ObservedFrequency": b} for a, b in zip(mp, ft)]
    cal = pd.DataFrame(rows).round(4); save(cal, "S9_calibration.csv"); save(pd.DataFrame(curves).round(4), "figure4_calibration_curves.csv")
    plt.figure(figsize=(5.5, 5.5)); plt.plot([0, 1], [0, 1], "--", color="gray", label="Perfect")
    for k, p in variants.items():
        ft, mp = calibration_curve(y_test, p, n_bins=10, strategy="quantile"); plt.plot(mp, ft, marker="o", label=k)
    plt.xlabel("Mean predicted probability"); plt.ylabel("Observed frequency"); plt.legend(); plt.title("Reliability (quantile bins, test set)")
    plt.tight_layout(); plt.savefig(FIG / "figure4_reliability.png", dpi=300); plt.show(); display(cal)

## 6. RAID — Resampling-Aware Inflation Decomposition (Tables 11–12, Figure 5, Tables S1–S3)
For each seed (0–29) and classifier (no class weighting in any arm; identical preprocessing within a classifier):

| Arm | Training data | Evaluation data |
|---|---|---|
| **A** | Resample-before-split: preprocessing + SMOTE on all data, then 80/20 split of the resampled set | Resampled test fold (≈50/50, ≈41% synthetic) |
| **A_real_bal** | Same model as A | Only the **real** rows of A's test fold, reweighted to 50/50 |
| **A_real** | Same model as A | Only the real rows of A's test fold (natural prevalence) |
| **A′ (submitted definition)** | Real training rows of the honest split + **all** synthetic rows | Honest test rows (identical to B) |
| **B** | Honest: SMOTE inside the training pipeline | Honest test split, natural prevalence |

Equation 8: **A − B = (A − A_real_bal) + (A_real_bal − A_real) + (A_real − B)** =
synthetic test points + prevalence shift + training contamination and test sampling.
Every finished (seed, classifier) pair is appended to `raid_runs.csv`; re-running skips finished pairs.

In [ ]:
def raid_model(name, seed):
    if name == "XGBoost": return xgb(TUNED, None, seed)
    if name == "RandomForest": return RandomForestClassifier(n_estimators=300, random_state=seed, n_jobs=-1)
    if name == "CatBoost": return CatBoostClassifier(iterations=300, depth=6, learning_rate=0.05, verbose=0, random_seed=seed, thread_count=-1)
    if name == "LogisticRegression": return LogisticRegression(max_iter=2000, random_state=seed)
    raise ValueError(name)

def raid_one(name, s):
    scale = name == "LogisticRegression"; n = len(y); yv = y.to_numpy(); out = []
    itr, ite = train_test_split(np.arange(n), test_size=TEST_SIZE, stratify=yv, random_state=s)
    B = ImbPipeline([("p", prep(scale=scale)), ("s", SMOTE(random_state=s)), ("m", raid_model(name, s))]).fit(X.iloc[itr], yv[itr])
    out.append(("B", metrics(yv[ite], B.predict_proba(X.iloc[ite])[:, 1])))
    Xall = prep(scale=scale).fit_transform(X); Xall = Xall.toarray() if sparse.issparse(Xall) else np.asarray(Xall)
    Xr, yr = SMOTE(random_state=s).fit_resample(Xall, yv)          # originals first, synthetic rows appended
    ptr, pte = train_test_split(np.arange(len(yr)), test_size=TEST_SIZE, stratify=yr, random_state=s)
    mA = raid_model(name, s).fit(Xr[ptr], yr[ptr]); pA = mA.predict_proba(Xr[pte])[:, 1]
    out.append(("A", metrics(yr[pte], pA)))
    real = pte < n; yR, pR = yr[pte][real], pA[real]
    w = np.where(yR == 1, 0.5 / yR.mean(), 0.5 / (1 - yR.mean()))
    out.append(("A_real_bal", metrics(yR, pR, w=w))); out.append(("A_real", metrics(yR, pR)))
    keep = np.ones(len(yr), bool); keep[ite] = False                  # drop real honest-test rows from training
    mAp = raid_model(name, s).fit(Xr[keep], yr[keep])
    out.append(("A_prime", metrics(yv[ite], mAp.predict_proba(Xall[ite])[:, 1])))
    extra = {"A_test_synthetic_share": float((pte >= n).mean()), "A_real_n": int(real.sum()), "A_real_prevalence": float(yR.mean()),
             "A_synthetic_pos_recall": float((pA[~real] >= .5).mean())}
    return [{"seed": s, "classifier": name, "arm": a, **m, **extra} for a, m in out]

RAID_FILE = OUT_DIR / "raid_runs.csv"
if RUN["raid"]:
    done = set()
    if RAID_FILE.exists():
        prev = pd.read_csv(RAID_FILE); done = set(zip(prev.seed, prev.classifier))
    for name in RAID_MODELS:
        for s in RAID_SEEDS:
            if (s, name) in done: continue
            t0 = time.time(); rows = pd.DataFrame(raid_one(name, s))
            rows.to_csv(RAID_FILE, mode="a", header=not RAID_FILE.exists(), index=False)
            print(f"{name} seed {s} done in {time.time() - t0:.0f}s")

In [ ]:
# 8b. Summarise RAID: mean, SD, 2.5-97.5 percentile, share of total, and sign stability per component
if RUN["raid"] and RAID_FILE.exists():
    runs = pd.read_csv(RAID_FILE).drop_duplicates(["seed", "classifier", "arm"])
    comps = {"Total (A - B)": ("A", "B"), "Synthetic test points (A - A_real_bal)": ("A", "A_real_bal"),
             "Prevalence shift (A_real_bal - A_real)": ("A_real_bal", "A_real"),
             "Contamination + test sampling (A_real - B)": ("A_real", "B"),
             "Submitted: A - A'": ("A", "A_prime"), "Submitted: A' - B": ("A_prime", "B")}
    rows = []
    for (clf, m), g in runs.melt(id_vars=["seed", "classifier", "arm"], value_vars=["F1", "ROC_AUC", "PR_AUC"], var_name="metric").groupby(["classifier", "metric"]):
        w = g.pivot_table(index="seed", columns="arm", values="value"); total = w["A"] - w["B"]
        for comp, (a, b) in comps.items():
            d = w[a] - w[b]
            rows.append({"classifier": clf, "metric": m, "component": comp, "n_seeds": len(d), "mean": d.mean(), "sd": d.std(),
                         "p2.5": d.quantile(.025), "p97.5": d.quantile(.975), "share_of_total_%": 100 * (d / total).mean(),
                         "seeds_positive_%": 100 * (d > 0).mean()})
    raid_sum = pd.DataFrame(rows).round(4); save(raid_sum, "table12_raid_decomposition.csv")
    arms = runs.groupby(["classifier", "arm"])[["F1", "ROC_AUC", "PR_AUC"]].agg(["mean", "std"]).round(4)
    arms.columns = [f"{a}_{b}" for a, b in arms.columns]; save(arms.reset_index(), "table11_raid_arms.csv")
    # Figure 5 - stacked F1 components per classifier; whisker = 2.5-97.5 percentile of the total over seeds
    cols = [("Synthetic test points (A - A_real_bal)", "#2a78d6"), ("Prevalence shift (A_real_bal - A_real)", "#eb6834"),
            ("Contamination + test sampling (A_real - B)", "#1baf7a")]
    order = [("XGBoost", "XGBoost (tuned)"), ("CatBoost", "CatBoost"), ("RandomForest", "Random Forest"), ("LogisticRegression", "Logistic Regression")]
    f1s = raid_sum[raid_sum.metric == "F1"]; fig, ax = plt.subplots(figsize=(6.6, 3.0), dpi=150)
    for i, (key, lab) in enumerate(order):
        if key not in set(f1s.classifier): continue
        left = 0
        for comp, colr in cols:
            r = f1s[(f1s.classifier == key) & (f1s.component == comp)].iloc[0]
            ax.barh(i, r["mean"], left=left, color=colr, height=0.56, edgecolor="white", linewidth=1.2)
            if r["mean"] > 0.05:
                ax.text(left + r["mean"] / 2, i, f"{r['share_of_total_%']:.0f}%", ha="center", va="center", color="white", fontsize=8, fontweight="bold")
            left += r["mean"]
        w = runs[runs.classifier == key].pivot_table(index="seed", columns="arm", values="F1"); d = w["A"] - w["B"]
        ax.errorbar(d.mean(), i, xerr=[[d.mean() - d.quantile(.025)], [d.quantile(.975) - d.mean()]], fmt="none", ecolor="k", elinewidth=0.9, capsize=3)
        ax.text(d.quantile(.975) + 0.006, i, f"+{d.mean():.3f}", va="center", fontsize=8)
    ax.set_yticks(range(len(order))); ax.set_yticklabels([o[1] for o in order]); ax.invert_yaxis(); ax.set_xlim(0, 0.37)
    ax.set_xlabel(f"Mean F1 inflation, arm A - arm B ({runs.seed.nunique()} seeds); whisker = 2.5-97.5 percentile of the total", fontsize=8)
    for sp in ["top", "right"]: ax.spines[sp].set_visible(False)
    ax.grid(axis="x", color="#e6e6e6", linewidth=0.6); ax.set_axisbelow(True)
    ax.legend([plt.Rectangle((0, 0), 1, 1, color=c) for _, c in cols], [c for c, _ in cols], loc="upper center",
              bbox_to_anchor=(0.45, -0.22), ncol=1, frameon=False, fontsize=7.5)
    plt.tight_layout(); plt.savefig(FIG / "figure5_raid_components.png", dpi=300, bbox_inches="tight"); plt.show()
    display(raid_sum[raid_sum.metric == "F1"])

## 7. Deduplicated data (Supplementary Table S6)
The 125 exact duplicate rows are removed before splitting and the headline models are re-run.

In [ ]:
if RUN["dedup"]:
    dd = df.drop_duplicates().reset_index(drop=True); Xd, yd = dd.drop(columns=[TARGET]), dd[TARGET]
    Xdt, Xde, ydt, yde = train_test_split(Xd, yd, test_size=TEST_SIZE, stratify=yd, random_state=SEED)
    key = lambda Z: Z.astype(str).agg("|".join, axis=1)
    overlap = int(key(X_test).isin(set(key(X_train))).sum())
    rows = []
    for name, pipe in {"Reference: tuned XGBoost, all features": Pipeline([("p", prep()), ("m", xgb(TUNED, spw_of(ydt)))]),
                       "Tuned XGBoost + FS (final)": Pipeline([("p", prep()), ("fs", SelectKBest(mutual_info_classif, k=20)), ("m", xgb(TUNED, spw_of(ydt)))]),
                       "Untuned XGBoost": Pipeline([("p", prep()), ("m", xgb(BASE, spw_of(ydt)))])}.items():
        pr = pipe.fit(Xdt, ydt).predict_proba(Xde)[:, 1]; rows.append({"Setting": name + " (deduplicated)", **metrics(yde, pr)})
    ded = pd.DataFrame(rows); ded[METRICS] = ded[METRICS].round(4); save(ded, "S6_dedup.csv")
    info = pd.DataFrame([{"rows": len(df), "duplicate_rows_removed": len(df) - len(dd), "rows_after": len(dd),
                          "duplicates_with_Revenue_1": int(df[df.duplicated(keep=False)][TARGET].sum()),
                          "original_test_rows_with_exact_twin_in_train": overlap}])
    save(info, "S6_dedup_info.csv"); display(info); display(ded)

## 8. Download (Colab only)

In [ ]:
if IN_COLAB:
    import shutil
    z = shutil.make_archive(str(OUT_DIR), "zip", OUT_DIR); print("archive:", z)
    from google.colab import files; files.download(z)